# Fase 1 · M00: Índice Transformación

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 1 — Transformación |
| **Módulo** | M00 — Índice |

---

## 🎯 Qué hace

Genera la página índice HTML de Fase 1 con la lista de módulos y su descripción.

## 📋 Requisitos

- `src/html/render.py` con `render_pagina_desde_fichero()`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase1/fase1_index.html` | Índice navegable de Fase 1 |

## 🔄 Flujo

```
render_pagina_desde_fichero() → HTML → escribe en disco
```

## ➡️ Siguiente

`f1_m02_limpieza.ipynb` — limpieza de datos originales


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
#
# Detecta entorno (Colab/Local), configura ROOT buscando 'src/',
# importa módulos del proyecto.
# Las rutas vienen TODAS de src.config — sin hardcodes.
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(
        f"No se encontró carpeta 'src/' subiendo desde {start}. "
        f"Verifica que el notebook está dentro de AU_UJI/"
    )

ROOT = _encontrar_root(Path.cwd())


sys.path.insert(0, str(ROOT))

# --- Imports del proyecto (rutas desde src.config, NUNCA hardcodeadas) ---
from src.config import (
    RUTA_RAW, RUTA_PROCESSED, RUTA_HTML,
    EXCEL_PRINCIPAL, EXCEL_PREINSCRIPCION,
    info_entorno
)
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_tarjetas_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

# --- Ruta de salida HTML ---
RUTA_FASE1 = RUTA_HTML / 'fase1'
crear_directorios([RUTA_FASE1])

info_entorno()

✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: DATOS ORIGINALES (sin tratar — siempre disponibles)
# ============================================================================
# Lee el Excel con openpyxl. Si el fichero está bloqueado por OneDrive o Excel,
# usa fallback con valores por defecto en lugar de romper la ejecución.
# ============================================================================

import shutil, tempfile
import pandas as pd
from openpyxl import load_workbook

fmt = formato_numero_es

def _leer_excel_seguro(ruta):
    """Lee un Excel copiándolo a temporal para evitar bloqueos de OneDrive/Excel."""
    try:
        with tempfile.NamedTemporaryFile(suffix='.xlsx', delete=False) as tmp:
            ruta_tmp = Path(tmp.name)
        shutil.copy2(ruta, ruta_tmp)
        wb = load_workbook(ruta_tmp, read_only=True)
        info = []
        for nombre in wb.sheetnames:
            ws = wb[nombre]
            info.append({
                'hoja': nombre,
                'filas': (ws.max_row - 1) if ws.max_row else 0,
                'cols': ws.max_column or 0
            })
        wb.close()
        ruta_tmp.unlink(missing_ok=True)
        return info, True
    except Exception as e:
        print(f'⚠️  No se pudo leer {ruta.name}: {e}')
        return [], False

hojas_info = []
total_registros_origen = 0
total_columnas_origen = 0

# Excel principal
info_principal, ok1 = _leer_excel_seguro(EXCEL_PRINCIPAL)
for h in info_principal:
    hojas_info.append(h)
    total_registros_origen += h['filas']
    total_columnas_origen += h['cols']
n_tablas = len(info_principal)

# Preinscripción
if EXCEL_PREINSCRIPCION.exists():
    info_pre, ok2 = _leer_excel_seguro(EXCEL_PREINSCRIPCION)
    for h in info_pre:
        hojas_info.append({'hoja': 'Preinscripción', 'filas': h['filas'], 'cols': h['cols']})
        total_registros_origen += h['filas']
        total_columnas_origen += h['cols']
    n_tablas += len(info_pre)

print('=' * 60)
print('DATOS ORIGINALES (sin tratar)')
print('=' * 60)
if hojas_info:
    for h in hojas_info:
        print(f"  📋 {h['hoja']:30s} → {fmt(h['filas']):>10s} filas × {h['cols']} cols")
    print(f'\n  📊 Total: {n_tablas} tablas, {fmt(total_registros_origen)} registros')
else:
    print('  ⚠️  Excel bloqueado o no encontrado — usando valores por defecto')
    n_tablas = 5
    total_registros_origen = 0
    total_columnas_origen = 0


⚠️  No se pudo leer datos_proyecto_sin_preinscrip.xlsx: [Errno 13] Permission denied: 'C:\\Users\\mjmor\\OneDrive - Universitat Jaume I\\2.- AU_UJI\\data\\00_raw\\datos_proyecto_sin_preinscrip.xlsx'
⚠️  No se pudo leer preinscripcion_si.xlsx: [Errno 13] Permission denied: 'C:\\Users\\mjmor\\OneDrive - Universitat Jaume I\\2.- AU_UJI\\data\\00_raw\\preinscripcion_si.xlsx'
DATOS ORIGINALES (sin tratar)
  ⚠️  Excel bloqueado o no encontrado — usando valores por defecto


In [3]:
# ============================================================================
# CELDA 3: DATOS FINALES (tras transformación — si están disponibles)
# ============================================================================
#
# Si df_alumno.parquet existe (Fase 1 ya ejecutada), calcula:
#   - Registros finales, columnas, alumnos únicos
# Si no existe, muestra 'Pendiente'.
# ============================================================================

ruta_alumno = RUTA_PROCESSED / 'df_alumno.parquet'

if ruta_alumno.exists():
    df_alumno = pd.read_parquet(ruta_alumno)
    n_registros_final = len(df_alumno)
    n_columnas_final = len(df_alumno.columns)
    n_alumnos = None
    for col in ['per_id_ficticio', 'nip', 'alumno_id']:
        if col in df_alumno.columns:
            n_alumnos = df_alumno[col].nunique()
            break
    del df_alumno

    print('=' * 60)
    print('DATOS FINALES (tras transformación)')
    print('=' * 60)
    print(f'  🎯 Registros: {fmt(n_registros_final)}')
    print(f'  📊 Columnas:  {n_columnas_final}')
    print(f'  👤 Alumnos:   {fmt(n_alumnos) if n_alumnos else "N/A"}')

    datos_finales_disponibles = True
else:
    n_registros_final = None
    n_columnas_final = None
    n_alumnos = None
    datos_finales_disponibles = False

    print('⚠ df_alumno.parquet no encontrado — ejecuta Fase 1 completa')
    print(f'  Ruta esperada: {ruta_alumno}')

DATOS FINALES (tras transformación)
  🎯 Registros: 109.568
  📊 Columnas:  37
  👤 Alumnos:   30.872


In [4]:
# ============================================================================
# CELDA 4: CONFIGURAR KPIs Y MÓDULOS
# ============================================================================

# --- KPIs: Datos originales (siempre visibles) ---
KPIS_ORIGEN = [
    {'valor': str(n_tablas), 'titulo': 'Tablas origen'},
    {'valor': fmt(total_registros_origen), 'titulo': 'Registros origen'},
    {'valor': str(total_columnas_origen), 'titulo': 'Columnas origen'},
]

# --- KPIs: Datos finales (o pendientes) ---
if datos_finales_disponibles:
    KPIS_FINAL = [
        {'valor': fmt(n_registros_final), 'titulo': 'Registros finales'},
        {'valor': str(n_columnas_final), 'titulo': 'Columnas finales'},
        {'valor': fmt(n_alumnos) if n_alumnos else 'N/A', 'titulo': 'Alumnos únicos'},
    ]
else:
    KPIS_FINAL = [
        {'valor': 'Pendiente', 'titulo': 'Registros finales'},
        {'valor': 'Pendiente', 'titulo': 'Columnas finales'},
        {'valor': 'Pendiente', 'titulo': 'Alumnos únicos'},
    ]

# --- Módulos de la fase ---
MODULOS = [
    {
        'archivo': 'm01_reportes_raw.html',
        'emoji': '📋',
        'titulo': 'Reportes Raw',
        'desc': f'{n_tablas} reportes Sweetviz de los datos originales antes de cualquier transformación.',
        'color': '#3182ce'
    },
    {
        'archivo': 'm02_limpieza.html',
        'emoji': '🧹',
        'titulo': 'Limpieza',
        'desc': 'Transformaciones aplicadas: snake_case, NaN, tipos de datos por cada tabla.',
        'color': '#38a169'
    },
    {
        'archivo': 'm03_reportes_clean.html',
        'emoji': '✨',
        'titulo': 'Reportes Clean',
        'desc': f'{n_tablas} reportes Sweetviz de los datos limpios. Comparación antes/después.',
        'color': '#3182ce'
    },
    {
        'archivo': 'm04_dataset_final.html',
        'emoji': '🎯',
        'titulo': 'Dataset Final',
        'desc': f'Unión de las {n_tablas} tablas en df_alumno. Variables derivadas: vive_fuera, tiene_beca.',
        'color': '#805ad5'
    },
    {
        'archivo': 'm05_dashboard.html',
        'emoji': '📊',
        'titulo': 'Dashboard',
        'desc': 'Dashboard interactivo con resumen visual del proceso de transformación.',
        'color': '#ed8936'
    },
    {
        'archivo': 'm06_grafo.html',
        'emoji': '🕸️',
        'titulo': 'Grafo',
        'desc': 'Grafo interactivo con nodos arrastrables mostrando flujo de tablas a df_alumno.',
        'color': '#ed8936'
    },
]

print(f'✅ Configuración: {len(KPIS_ORIGEN)} KPIs origen + {len(KPIS_FINAL)} KPIs finales + {len(MODULOS)} módulos')

✅ Configuración: 3 KPIs origen + 3 KPIs finales + 6 módulos


In [5]:
# ============================================================================
# CELDA 5: GENERAR HTML
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- Navegación ---

# --- Sección: Resumen ---
texto_resumen = (
    '<p>'
    f'Limpieza, transformación y unificación de <strong>{n_tablas} tablas</strong> '
    f'({fmt(total_registros_origen)} registros originales) '
    'provenientes de 2 archivos Excel en un único dataset <strong>df_alumno</strong>.'
    '</p>'
    '<p>'
    'Incluye generación de reportes Sweetviz (antes/después), documentación de '
    'transformaciones y grafos interactivos de trazabilidad.'
    '</p>'
)
seccion_resumen = generar_seccion_html(
    titulo='Resumen de la Fase',
    contenido=texto_resumen,
    icono='📥'
)

# --- Sección: KPIs Datos Originales ---
kpis_origen_html = generar_kpis_html(KPIS_ORIGEN)
seccion_origen = generar_seccion_html(
    titulo='Datos Originales (sin tratar)',
    contenido=kpis_origen_html,
    icono='📋'
)

# --- Sección: KPIs Datos Finales ---
kpis_final_html = generar_kpis_html(KPIS_FINAL)
seccion_final = generar_seccion_html(
    titulo='Datos Finales (tras transformación)',
    contenido=kpis_final_html,
    icono='🎯'
)

# --- Sección: Módulos ---
modulos_contenido = generar_tarjetas_html([
    {
        'titulo': m['titulo'],
        'descripcion': m['desc'],
        'emoji': m['emoji'],
        'link': m['archivo'],
        'link_texto': 'Abrir módulo →',
        'color': m['color']
    }
    for m in MODULOS
])
seccion_modulos = generar_seccion_html(
    titulo='Módulos',
    contenido=modulos_contenido,
    icono='📦'
)

# --- Contenido completo ---
contenido_html = seccion_resumen + seccion_origen + seccion_final + seccion_modulos

# --- Página completa ---
html_completo = render_pagina_desde_fichero(
    'f1_m00_indice.ipynb',
    contenido_html,
    carpeta_notebook='fase1_transformacion'
)

# --- Guardar ---
ruta_html = RUTA_FASE1 / 'fase1_index.html'
guardar_html(html_completo, ruta_html)

print(f'\n✅ HTML generado: {ruta_html}')

GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\fase1_index.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\fase1_index.html


In [6]:
# ============================================================================
# CELDA 6: RESUMEN
# ============================================================================

print('\n' + '=' * 60)
print('✅ F1-M00 COMPLETADO')
print('=' * 60)
print(f'\n📄 Archivo generado: {ruta_html}')
print(f'\n📋 Datos originales: {n_tablas} tablas, {fmt(total_registros_origen)} registros')
if datos_finales_disponibles:
    print(f'🎯 Datos finales:    {fmt(n_registros_final)} registros, {n_columnas_final} columnas, {fmt(n_alumnos)} alumnos')
else:
    print(f'⚠  Datos finales:    Pendiente (ejecuta Fase 1 completa)')
print(f'\n📌 Siguiente: f1_m01_reportes_raw.ipynb')
print('=' * 60)


✅ F1-M00 COMPLETADO

📄 Archivo generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\fase1_index.html

📋 Datos originales: 5 tablas, 0 registros
🎯 Datos finales:    109.568 registros, 37 columnas, 30.872 alumnos

📌 Siguiente: f1_m01_reportes_raw.ipynb
